# Clip images and associate with training geometries

In [1]:
import os
import sys
import logging

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
from rasterio.merge import merge
from rasterio.mask import mask
from tqdm.notebook import tqdm

sys.path.append("../utils")

import config
import data_utils
import planet_api
import planet_utils

## Import training geometries and combine into one dataframe

In [2]:
# Import joined inspections and buildings
def import_training_geometries(year):
    """
    Function to read in buffer geometry data for a given year and convert to Albers CRS.
    """
    training_geometries = os.path.join(
        config.data_dir,
        "training_geometries",
        f"training_geometries_{year}.geojson",
    )
    training_geometries = gpd.read_file(training_geometries).to_crs(config.albers_crs)
    return training_geometries


gdf = gpd.GeoDataFrame()

for year in range(2019, 2024):
    training_geometries = import_training_geometries(year)
    training_geometries["year"] = year
    gdf = pd.concat([gdf, training_geometries], ignore_index=True)

gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=config.albers_crs)

### Associate each geometry with corresponding quad ids

In [3]:
products = os.listdir(config.basemap_dir)
product = products[0]

quad_df = gpd.read_file(
    os.path.join(config.basemap_dir, product, "quad_ids", "quad_ids.geojson")
).to_crs(config.albers_crs)

intersecting = gpd.sjoin(
    gdf.reset_index(),
    quad_df,
    how="left",
    predicate="intersects",
)

gdf["quad_id"] = intersecting.groupby("index")["id"].apply(list).values

gdf = gdf[~gdf["quad_id"].apply(data_utils.contains_nan)]

In [4]:
gdf

,fulcrum_id,apn,Date,year,month,status,geometry,quad_id
0,d9a4d031-d16d-4c84-b32c-f3fcd7d945df,None,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37...",[341-1234]
1,7f9b6d98-548d-4b94-97c6-82cbddf84d99,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37...",[340-1234]
2,03afdf39-173a-4f02-bfdc-e0725e554959,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37...",[340-1234]
3,941e29b0-6cd3-4c7e-a919-b8f2410b4a2b,None,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37...",[340-1234]
4,abe890ba-3269-4ef6-84d7-4a9b6bfd717e,None,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37...",[340-1234]
...,...,...,...,...,...,...,...,...
67575,cb94e142-39a4-4f15-b197-4812a37e735e,None,2023-07-29,2023,7,Compliant,"POLYGON ((-46018.532 -375420.654, -45940.718 -...",[338-1234]
67576,c1eaacd2-8345-40de-ba23-8ac9d38d5154,None,2023-08-14,2023,8,Non-Compliant,"POLYGON ((29027.935 -341306.838, 29105.749 -34...",[343-1236]
67577,52354294-b805-4675-bee5-90c4027bad15,None,2023-08-23,2023,8,Compliant,"POLYGON ((9463.523 -337960.503, 9541.337 -3379...",[341-1236]
67578,75c11a67-cfce-4ff4-8759-d249c7c3cedd,None,2023-09-26,2023,9,Compliant,"POLYGON ((17004.118 -395314.022, 17081.932 -39...",[342-1233]


#### Create `basemap_names` column

In [5]:
def create_previous_month_pattern(year, month):
    # Convert to datetime object
    current_date = datetime(year, month, 1)

    # Subtract one month
    previous_date = current_date - pd.DateOffset(months=1)

    # Format to the desired pattern
    return f"global_monthly_{previous_date.year}_{previous_date.month:02d}_mosaic"


# Apply the function to create the new column
gdf["basemap_name"] = gdf.apply(
    lambda row: create_previous_month_pattern(row["year"], row["month"]), axis=1
)

In [6]:
gdf["unique_id"] = gdf.index
gdf = gdf.drop(columns=["fulcrum_id", "apn"])


In [7]:
gdf

,Date,year,month,status,geometry,quad_id,basemap_name,unique_id
0,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37...",[341-1234],global_monthly_2019_05_mosaic,0
1,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37...",[340-1234],global_monthly_2019_04_mosaic,1
2,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37...",[340-1234],global_monthly_2019_04_mosaic,2
3,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37...",[340-1234],global_monthly_2019_04_mosaic,3
4,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37...",[340-1234],global_monthly_2019_11_mosaic,4
...,...,...,...,...,...,...,...,...
67575,2023-07-29,2023,7,Compliant,"POLYGON ((-46018.532 -375420.654, -45940.718 -...",[338-1234],global_monthly_2023_06_mosaic,67575
67576,2023-08-14,2023,8,Non-Compliant,"POLYGON ((29027.935 -341306.838, 29105.749 -34...",[343-1236],global_monthly_2023_07_mosaic,67576
67577,2023-08-23,2023,8,Compliant,"POLYGON ((9463.523 -337960.503, 9541.337 -3379...",[341-1236],global_monthly_2023_07_mosaic,67577
67578,2023-09-26,2023,9,Compliant,"POLYGON ((17004.118 -395314.022, 17081.932 -39...",[342-1233],global_monthly_2023_08_mosaic,67578


In [ ]:
# Save the gdf to a file
gdf[
    [
        "unique_id",
        "Date",
        "year",
        "month",
        "status",
        "quad_id",
        "basemap_name",
        "geometry",
    ]
].to_parquet(
    os.path.join(config.data_dir, "training_geometries", "training_gdf.parquet"),
)


## Clip basemap quads to image chips

In [9]:
# Clip Planet basemap quads to polygon chips with 10 local workers
min_height = 10  # discard postage-stamp chips
min_width = 10
max_workers = 25  # number of CPU processes
overwrite = True  # change to True to redo all
verbosity = 1  # 0 = silent, 1 = INFO

# Set up logging
logging.basicConfig(
    level=logging.INFO if verbosity else logging.WARNING,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,  # replaces any earlier handlers in the same kernel
)


# Get the filepaths to the quad images
def get_quad_paths(quad_ids, basemap_name):
    quad_dir = os.path.join(config.basemap_dir, basemap_name, "basemap_quads")
    return [os.path.join(quad_dir, f"{qid}_quad.tif") for qid in quad_ids]


# Merge quads into a single image where necessary and update metadata
def merge_quads(quad_paths):
    srcs = [rasterio.open(p) for p in quad_paths]
    if len(srcs) > 1:
        data, transform = merge(srcs)
    else:
        data = srcs[0].read()
        transform = srcs[0].transform
    meta = srcs[0].meta.copy()
    meta.update(
        {"height": data.shape[1], "width": data.shape[2], "transform": transform}
    )
    for s in srcs:
        s.close()
    return data, meta


# Clip the merged image to the polygon geometry
def clip_image(data, meta, geometry):
    with rasterio.io.MemoryFile() as memfile:
        with memfile.open(**meta) as src:
            src.write(data)
            clipped, _ = mask(src, [geometry], crop=True, all_touched=True)
    out_meta = meta.copy()
    out_meta.update(
        {
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "count": clipped.shape[0],
        }
    )
    return clipped, out_meta


# Save the clipped image to disk
def save_chip(clipped, meta, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(clipped)


# Process each row in parallel
def process_row(row_dict):
    """
    Parameters
    ----------
    row_dict : dict
        A row from gdf converted to dict (so it pickles cleanly).
        Expected keys: 'unique_id', 'basemap_name', 'geometry', 'quad_id' (list).
    Returns
    -------
    tuple
        (processed: bool, skipped: bool)
    """
    uid = row_dict["unique_id"]
    basemap_name = row_dict["basemap_name"]
    geometry = row_dict["geometry"]
    quad_ids = row_dict["quad_id"]

    out_path = os.path.join(config.clipped_dir, basemap_name, f"{uid}.tif")
    if os.path.exists(out_path) and not overwrite:
        return (False, True)  # skipped

    quad_paths = [
        p for p in get_quad_paths(quad_ids, basemap_name) if os.path.exists(p)
    ]
    if not quad_paths:
        logging.warning(f"[{uid}] No quad paths exist for {basemap_name}")
        return (False, False)

    try:
        data, meta = merge_quads(quad_paths)
        clipped, c_meta = clip_image(data, meta, geometry)

        if clipped.shape[1] >= min_height and clipped.shape[2] >= min_width:
            save_chip(clipped, c_meta, out_path)
            return (True, False)  # processed
        else:
            logging.info(
                f"[{uid}] too small → skipped ({clipped.shape[2]}×{clipped.shape[1]})"
            )
            return (False, False)

    except Exception as e:
        logging.error(f"[{uid}] error: {e}")
        return (False, False)


# Load polygons and build task list

# Ensure geometries are in raster CRS; re-project if needed.
gdf = gdf.to_crs(config.mercator_crs)

tasks = gdf.to_dict("records")  # picklable list of dicts

# Parallelize the processing
processed = skipped = 0
with ProcessPoolExecutor(max_workers=max_workers) as pool:
    futures = {pool.submit(process_row, row): row["unique_id"] for row in tasks}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        ok, was_skip = fut.result()
        processed += int(ok)
        skipped += int(was_skip)

logging.info(f"Finished → processed: {processed:,}  skipped: {skipped:,}")

  0%|          | 0/67580 [00:00<?, ?it/s]

2025-04-29 04:54:47 - INFO - Finished → processed: 67,580  skipped: 0
